<a href="https://colab.research.google.com/github/nishithy13/ds2002-fa26/blob/main/2026-10-16%20%E2%80%94%20Walmart_Warm_Up_SKU_Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))

poptarts = cat[cat['product_name'].str.lower().str.contains('pop')]

print(len(poptarts))
print(poptarts)

3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = cat[cat['product_name'].str.lower().str.contains('strawberry')]

print(strawberry[['sku', 'product_name']])

print(poptarts[~poptarts['product_name'].str.lower().str.contains('strawberry')])

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
Empty DataFrame
Columns: [sku, product_name, category]
Index: []


**Pop-Tart rows that are a different product:** None were found in the current dataset. Since all the poptart names contain strawberry, they can all be referenced as strawberry poptart products.  

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'}

cat['canonical_sku'] = cat['sku'].map(mapping).fillna(cat['sku'])

cat.head()

,sku,product_name,category,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,PT-12
1,SKU#459812,Strawberry Pop-Tart,Breakfast,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print(cat['category'].value_counts())

cat['category'] = cat['category'].str.strip().str.title()

print(cat['category'].value_counts())

category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
cat['normalized_name'] = cat['product_name'].str.lower().str.replace(',', '').str.replace('-', '')

groups = cat.groupby('normalized_name')['sku'].count()

print(groups[groups > 1])

normalized_name
poptarts strawberry    2
Name: sku, dtype: int64


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
rows_before = len(cat)

assert cat['canonical_sku'].isna().sum() == 0
assert len(cat) == rows_before

print("Mapping is safe")

Mapping is safe


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families beyond Pop-Tarts that may be fragmented are the products with different capitalization or punctuation in their names. I would group products together when their names are refering to the same product, even if the formatting is different. For instance, I would make the names lowercase and remove unnecessary punctuation before comparing them.

b) Consolidation becomes a business decision when it is not clear whether two products are actually the same product. For instance, two products with similar names but different sizes might need to stay separate becdue to the reason that customers purchase them as different items. I would ask the Walmart product catalog manager to confirm whether they should be grouped together.
